## Overview: Lab 8 - Extending Local Search Techniques

> In the previous lab, you were introduced to the fundamental concepts of **Local Search Algorithms**, specifically focusing on **Steepest Ascent Hill Climbing**. You learned how these algorithms differ from traditional search strategies by exploring the state space locally, without maintaining a full search tree or queue.
>
> This lab builds upon that foundation by introducing several advanced local search techniques. We will explore how these methods enhance the basic hill-climbing approach, offering improved capabilities to navigate complex search spaces and avoid local optima.
>
> We will continue to apply these strategies to two classic AI problems:
>
> -   **The Traveling Salesperson Problem (TSP)**: Finding the shortest possible route that visits each city exactly once and returns to the starting city in a complete graph.
> -   **The 8-Queens Problem**: Placing eight queens on a chessboard such that no two queens threaten each other.

## Objectives

> -   **Extend Local Search Implementations**: Add to the existing steepest ascent implementation by incorporating stochastic hill climbing, first choice hill climbing, random restart hill climbing, and simulated_annealing.
> -   **Implement and Evaluate Advanced Local Search Strategies**:
>     -   **Stochastic Hill Climbing (HC)**: Randomly select among neighbors with better evaluations.
>     -   **First Choice Hill Climbing**: Select the first neighbor that improves the current state.
>     -   **Random Restart Hill Climbing**: Perform multiple hill-climbing searches from random initial states.
>     -   **Simulated annealing**: probabilistically accept worse moves to escape local optima.
> -   **Analyze and Compare Algorithm Performance**: Evaluate the effectiveness of each strategy on TSP and the 8-Queens Problem.
> -   **Understand the Trade-offs**: Discuss the strengths and weaknesses of each local search method, including their ability to escape local optima and the computational costs involved.

## 🛠️ Your Tasks:

### ✅ 1. Complete the Following Methods:

> -   Ensure the Traveling Salesperson Problem (TSP) class and Eight Queens Problem class from the previous lab are correctly implemented.
> -   Enhance the \`_hill_climbing\` method to include steepest, stochastic, and first choice hill climbing.
> -   Implement the \`random_restart_hill_climbing\` method.
> -   Implement the \`_simulated_annealing\` method.
> -   Implement the \`local_search\` method to dispatch the correct local search strategy.

### ✅ 2. Run the Provided Tests:

> Test your implementation on both:
> -   \`tsp_problem\` (for TSP)
> -   \`queens_problem\` (for 8-Queens)
>
> Ensure your tests cover all implemented strategies (Steepest, Stochastic, First Choice, Random Restart, Simulated Annealing).

### ✅ 3. Analyze the Results:

> -   Compare the outputs of each strategy to understand their performance characteristics and how they differ from the steepest ascent method you implemented in the previous lab.
> -   Analyze how each strategy explores the state space and whether it converges to a valid solution.
> -   Discuss the impact of parameters like \`initial_temperature\`, \`cooling_rate\`, \`max_iterations\`, and the number of restarts on the quality of the solution.
> -   Reflect on the trade-offs between exploration and exploitation in each strategy.
> -   Discuss the problems each strategy is best suited for, and how they improve upon the basic steepest descent method.

In [5]:
import random
import math
import copy

random.seed(42)

# ------------------------------------------------------------------------------
# Candidate Class: Represents a potential solution state and its evaluation
# ------------------------------------------------------------------------------
class Candidate:
    """
    Represents a candidate solution in a local search context.

    Attributes:
        state: The specific configuration of the solution (e.g., a tour permutation, queen positions).
        value: The evaluation score of the state (lower is generally better in minimization problems).
    """
    def __init__(self, state, value):
        self.state = state
        self.value = value

    def __repr__(self):
        # Provides a clear string representation of the candidate object
        # Truncate long states for readability
        state_repr = str(self.state)
        if len(state_repr) > 50:
            state_repr = state_repr[:47] + "..."
        return f"Candidate(state={state_repr}, value={self.value})"

# ------------------------------------------------------------------------------
# Problem Interface (Conceptual)
# Any problem used with these search algorithms should ideally provide:
#   - initial_state: A starting candidate solution state.
#   - evaluate(state): A function to calculate the cost/value of a given state.
#   - generate_neighbors(state): A function to produce neighboring states from a given state.
#   - (Optional) A way to generate a new random initial state.
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Traveling Salesperson Problem (TSP) Implementation
# ------------------------------------------------------------------------------
class TSPProblem:
    """
    Represents the Traveling Salesperson Problem.

    Finds the shortest tour visiting all cities exactly once and returning to the start.
    Assumes a complete graph where travel between any two cities is possible.

    Attributes:
        cities (dict): Maps city names to (x, y) coordinates.
        city_list (list): An ordered list of city names.
        initial_state (list): A randomly generated initial tour (permutation of cities).
    """
    def __init__(self, cities):
        """
        Initializes the TSP problem.

        Args:
            cities (dict): A dictionary mapping city names to coordinates.
                           Example: {'A': (0, 0), 'B': (1, 5), ...}
        """
        if not cities:
            raise ValueError("City dictionary cannot be empty.")
        self.cities = cities
        self.city_list = list(cities.keys())
        # Generate a random initial tour upon initialization
        self.initial_state = self._generate_random_state()

    def _generate_random_state(self):
        """
        Generates a new random tour (permutation of cities).

        Returns:
            list: A random permutation of city names.

        Example:
            >>> self.city_list = ['A', 'B', 'C']
            >>> problem._generate_random_state()
            ['B', 'C', 'A']
        """
        # TODO: Copy self.city_list, shuffle it, and return the new list
        return random.sample(self.city_list, len(self.city_list))

    def evaluate(self, state):
        """
        Calculates the total length of a given tour (state).

        Uses Euclidean distance and treats the tour as cyclic.

        Args:
            state (list): A permutation of city names representing the tour.

        Returns:
            float: The total distance (cost) of the tour.

        Example:
            >>> cities = {'A': (0,0), 'B': (3,4)}
            >>> problem = TSPProblem(cities)
            >>> problem.evaluate(['A', 'B'])
            10.0
        """
        # TODO: Handle edge cases (len(state) < 2)
        # TODO: Loop through each consecutive pair (including wrap-around) and sum distances via self._distance
        if len(state)<2 : 
            return 0.0
        total=0.0
        for i in range(len(state)):
            total+=self._distance(self.cities[state[i]], 
                                  self.cities[state[(i+1)%len(state)]])
        return total
    
    def _distance(self, coord1, coord2):
        """
        Calculates Euclidean distance between two coordinates.

        Args:
            coord1 (tuple): (x, y) of the first city.
            coord2 (tuple): (x, y) of the second city.

        Returns:
            float: Euclidean distance.

        Example:
            >>> problem._distance((0,0), (3,4))
            5.0
        """
        # TODO: Compute and return math.sqrt((x1-x2)**2 + (y1-y2)**2)
        return math.sqrt((coord1[0]-coord2[0])**2 + (coord1[1]-coord2[1])**2)

    def generate_neighbors(self, state):
        """
        Generates neighboring tours using a simple swap (2-opt) strategy.

        Args:
            state (list): The current tour (permutation of city names).

        Returns:
            list: A list of neighboring tour states.

        Example:
            >>> problem.generate_neighbors(['A', 'B', 'C'])
            [['B', 'A', 'C'], ['C', 'B', 'A'], ['A', 'C', 'B']]
        """
        # TODO: If len(state) < 2, return []
        # TODO: Iterate through all unique index pairs (i < j), swap, and collect neighbors
        if len(state)<2 : 
            return []
        neighbors=[]
        for i in range(len(state)-1):
            for j in range(i+1,len(state)):
                neighbor=copy.deepcopy(state)
                neighbor[i], neighbor[j] = (neighbor[j],neighbor[i])
                neighbors.append(neighbor) 
        return neighbors

# ------------------------------------------------------------------------------
# Eight Queens Problem Implementation
# ------------------------------------------------------------------------------
class EightQueensProblem:
    """
    Represents the 8-Queens problem using a permutation encoding.

    The goal is to place 8 queens on an 8x8 chessboard such that no two queens threaten each other.
    The state represents the row position of the queen in each column.

    Attributes:
        size (int): The size of the board (fixed at 8 by default).
        initial_state (list): A randomly generated initial placement (permutation).
    """
    def __init__(self, size=8, initial_state=None):
        """
        Initializes the 8-Queens problem.

        Args:
            size (int): The size of the chessboard (default 8).
            initial_state (list, optional): A specific starting permutation.
                                           If None, a random one is generated.
        """
        self.size = size
        if initial_state:
            if len(initial_state) != self.size or len(set(initial_state)) != self.size:
                 raise ValueError(f"Initial state must be a valid permutation of 0..{size-1}")
            self.initial_state = initial_state
        else:
            # Generate a random initial state if none provided
            self.initial_state = self._generate_random_state()


    def _generate_random_state(self):
        """
        Helper to generate a new random state (permutation of rows).

        Returns:
            list: A random permutation of range(self.size).

        Example:
            >>> eq._generate_random_state()
            [4, 0, 7, 2, 5, 1, 6, 3]
        """
        # TODO: Create a list(range(self.size)), shuffle it, and return
        state = list(range(self.size))
        random.shuffle(state)
        return state

    def evaluate(self, state):
        """
        Calculates the number of attacking pairs of queens (evaluation cost).

        Counts diagonal conflicts; row and column conflicts impossible by permutation encoding.

        Args:
            state (list): The current placement of queens (permutation).

        Returns:
            int: Number of pairs of queens attacking each other diagonally.

        Example:
            >>> eq.evaluate([0,1,2,3,4,5,6,7])
            28  # Worst-case all on diagonal
        """
        # TODO: Iterate over all pairs (i < j), check abs(row_i - row_j) == abs(i - j), count conflicts
        conflicts = 0
        for i in range(len(state)-1):
            for j in range(i+1, len(state)):
                if abs(state[i] - state[j]) == abs(i - j):
                    conflicts += 1
        return conflicts

    def generate_neighbors(self, state):
        """
        Generates neighboring states by swapping two queen positions.

        Args:
            state (list): The current queen placement state.

        Returns:
            list: A list of neighboring states.

        Example:
            >>> eq.generate_neighbors([0,1,2])
            [[1,0,2], [2,1,0], [0,2,1]]
        """
        # TODO: If len(state) < 2, return []
        # TODO: Swap each unique pair of indices and collect new states
        if len(state)<2 : 
            return []
        neighbors=[]
        for i in range(len(state)-1):
            for j in range(i+1, len(state)):
                neighbor=copy.deepcopy(state)
                neighbor[i], neighbor[j] = (neighbor[j],neighbor[i])
                neighbors.append(neighbor) 
        return neighbors

# ------------------------------------------------------------------------------
# Local Search Algorithms
# ------------------------------------------------------------------------------
def _steepest_ascent_search(problem):
    """
    Performs Steepest Ascent Hill Climbing (for minimization).

    Iteratively moves to the best neighboring state until no improvement.

    Args:
        problem: An object with initial_state, evaluate(), and generate_neighbors().

    Returns:
        Candidate: The best candidate found (local optimum).

    Example:
        >>> best = _steepest_ascent_search(tsp_problem)
    """

    # TODO: Implement Steepest Ascent Hill Climbing steps:
    # Initialization: Start with a randomly chosen initial solution.
    # Evaluation: Calculate a numerical value that represents the "goodness" of the current solution.
    # Repeat:
    #     Generate the neighbors states of the current solution.
    #     Evaluate each neighboring solution, calculating its "evaluation" value.
    #     Select the neighbor with the best "evaluation" value.
    #     If the best neighbor's value is better (lower) than the current solution's value, replace the current solution with the best neighbor.
    #     If no neighbor is better than the current solution, stop the iteration (break).
    # Result: Return the final solution, which is the best solution found during the search.
    init_state = problem.initial_state
    init_value = problem.evaluate(init_state)
    current = Candidate(init_state, init_value)

    while True:
        neighbors = problem.generate_neighbors(current.state)
        best_neighbor = None
        best_value = current.value 
        # Find the neighbor with the lowest cost
        for neighbor in neighbors:
            val = problem.evaluate(neighbor)
            if val < best_value:
                best_value = val
                best_neighbor = neighbor
        if best_neighbor is None:
            break
        current = Candidate(best_neighbor, best_value)
    return current

#Stochastic Hill Climbing
def _stochastic_hill_climbing(problem):
    """
    Performs Stochastic Hill Climbing (for minimization).

    Picks a random neighbor in each iteration. If it's better, move to it;
    otherwise, stop and return the current state.
    """
    init_state = problem.initial_state
    init_value = problem.evaluate(init_state)
    current = Candidate(init_state, init_value)

    while True:
        neighbors = problem.generate_neighbors(current.state)
        if not neighbors:
            break

        # Randomly choose one neighbor
        neighbor = random.choice(neighbors)
        value = problem.evaluate(neighbor)

        # If the neighbor is better, move to it
        if value < current.value:
            current = Candidate(neighbor, value)
        else:
            break  # No better move found
    return current

#First Choice HC
def _First_Choice_HC(problem):
    """
    Performs First Choice Hill Climbing (for minimization).

    Repeatedly picks the first better neighbor found and moves to it.
    Stops when no better neighbor is found.
    """
    init_state = problem.initial_state
    init_value = problem.evaluate(init_state)
    current = Candidate(init_state, init_value)

    while True:
        neighbors = problem.generate_neighbors(current.state)
        if not neighbors:
            break
        better = False
        for neighbor in neighbors:
            value = problem.evaluate(neighbor)
            if value < current.value:
                current = Candidate(neighbor, value)
                better = True
                break 
        if not better:
            break
    return current

#Random restart HC
def _Random_Restart_HC(problem, max_restarts=10):
    """
    Performs Random Restart Hill Climbing (for minimization).

    Repeatedly runs First Choice Hill Climbing from random initial states.
    Keeps the best solution found.
    """
    best_candidate = _First_Choice_HC(problem)

    for _ in range(max_restarts):
        problem.initial_state = problem._generate_random_state()  # set a new random start state
        candidate = _First_Choice_HC(problem)

        if candidate.value < best_candidate.value:
            best_candidate = candidate

    return best_candidate

#Simulated Annealing
def _simulated_annealing(problem, initial_temperature, cooling_rate, max_iterations):
    """
    Performs the simulated annealing search.

    Input:
      - problem: a problem instance which provides:
             * initial_state (a starting solution)
             * evaluate(state): a function that returns the evaluation (cost) of the state.
             * generate_random_neighbor(state): returns a randomly generated neighbor state.
      - initial_temperature: a float representing the starting temperature for the annealing schedule.
      - cooling_rate: a float between 0 and 1 that reduces the temperature at each iteration (e.g., 0.999).
      - max_iterations: an integer representing the maximum iterations to execute.

    Output:
      - Returns a Candidate representing the best solution found (state and its evaluation value).
    """
    # Step 1: Initialize the current state with problem.initial_state and compute its evaluation.
    current_state = problem.initial_state[:]  # Work with a copy
    current_value = problem.evaluate(current_state)
    current_candidate = Candidate(current_state, current_value)

    # Step 2: Set best_candidate equal to the current state as the starting best solution and temperature equal initial_temperature
    best_candidate = current_candidate  # Track the best solution overall

    temperature = initial_temperature

    # Step 3: Repeat until max_iterations is reached:
    #    a. If the temperature falls below a small threshold (e.g., 1e-6), stop the process (break).
    #    b. Generate one random neighbor using problem.generate_random_neighbor.
    #    c. Evaluate the generated neighbor's state.
    #    d. Calculate the change in evaluation: delta_e = neighbor_value - current_value.
    #    e. If the neighbor is better (delta_e < 0), move to the neighbor and update best_candidate if needed.
    #    f. If the neighbor is worse, accept it with a probability exp(-delta_e / temperature) (using math.exp).
    #    g. Reduce the temperature by multiplying it by cooling_rate for the next iteration.
    for _ in range(max_iterations):
        if temperature < 1e-6:  # Check if temperature is below threshold
            break
            
        # Generate a random neighbor
        neighbors = problem.generate_neighbors(current_state)
        if not neighbors:
            break
        # Randomly choose one neighbor
        neighbor_state = random.choice(neighbors)
        neighbor_value = problem.evaluate(neighbor_state)
        
        # Calculate change in evaluation
        delta_e = neighbor_value - current_value
        
        # Decide whether to accept the new solution
        if delta_e < 0:  # If the neighbor is better
            current_state = neighbor_state[:]
            current_value = neighbor_value
            current_candidate = Candidate(current_state, current_value)
            
            # Update best solution if needed
            if current_value < best_candidate.value:
                best_candidate = current_candidate
        else:
            # Accept worse solutions with a probability
            acceptance_probability = math.exp(-delta_e / temperature)
            if random.random() < acceptance_probability:
                current_state = neighbor_state[:]
                current_value = neighbor_value
                current_candidate = Candidate(current_state, current_value)
        
        # Reduce temperature for next iteration
        temperature *= cooling_rate

    # Step 4: After the loop completes, return the best_candidate found during the process.
    return best_candidate

def local_search(problem,
                 strategy="steepest",
                 num_restarts=10, base_restart_strategy="steepest",
                 initial_temperature_sa=50, cooling_rate_sa=0.9990, max_iterations_sa=5000):
    """
    Dispatches only Steepest Ascent for this lab.
    """
    if strategy == "steepest":
        return _steepest_ascent_search(problem)
    elif strategy == "stochastic" :
        return _stochastic_hill_climbing(problem)
    elif strategy == "first_choice":
        return _First_Choice_HC(problem)
    elif strategy == "random_restart":
        return _Random_Restart_HC(problem, num_restarts)
    elif strategy == "simulated":
        return _simulated_annealing(problem, initial_temperature_sa, cooling_rate_sa, max_iterations_sa)
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Choose 'steepest', 'stochastic', "
                         f"'first_choice', 'random_restart', or 'simulated_annealing'.")

# ------------------------------------------------------------------------------
# Test Functions
# ------------------------------------------------------------------------------
def test_tsp():
    print("\n----- TSP Local Search Test -----")

    cities = {
        'A': (0, 0), 'B': (2, 10), 'C': (5, 4), 'D': (6, 15),
        'E': (9, 3), 'F': (14, 10), 'G': (15, 0), 'H': (20, 5),
        'I': (22, 12), 'J': (25, 2), 'K': (28, 18), 'L': (30, 8),
        'M': (33, 15), 'N': (35, 3), 'O': (38, 10), 'P': (40, 0),
        'Q': (42, 7), 'R': (45, 14), 'S': (48, 5), 'T': (50, 16)
    }
    num_restarts_tsp = iterations_tsp = 50

    tsp_problem_base = TSPProblem(cities)  # Create one instance to show initial state
    print(f"Sample Initial TSP tour: {tsp_problem_base.initial_state}")
    print(f"Sample Initial tour cost: {tsp_problem_base.evaluate(tsp_problem_base.initial_state):.2f}")

    # --- Test Hill Climbing Variants ---
    candidate_steepest = local_search(TSPProblem(cities), strategy="steepest")
    print(f"Steepest Ascent Final tour cost: {candidate_steepest.value:.2f}")

    candidate_stochastic = local_search(TSPProblem(cities), strategy="stochastic")
    print(f"Stochastic HC Final tour cost: {candidate_stochastic.value:.2f}")

    candidate_first_choice = local_search(TSPProblem(cities), strategy="first_choice")
    print(f"First Choice HC Final tour cost: {candidate_first_choice.value:.2f}")

    # --- Test Random Restart ---
    candidate_restart_steepest = local_search(TSPProblem(cities), strategy="random_restart", num_restarts=num_restarts_tsp, base_restart_strategy="steepest")
    print(f"Random Restart (base: Steepest) Final tour cost: {candidate_restart_steepest.value:.2f}")

    candidate_restart_stochastic = local_search(TSPProblem(cities), strategy="random_restart", num_restarts=num_restarts_tsp, base_restart_strategy="stochastic")
    print(f"Random Restart (base: Stochastic) Final tour cost: {candidate_restart_stochastic.value:.2f}")

    candidate_restart_first_choice = local_search(TSPProblem(cities), strategy="random_restart", num_restarts=num_restarts_tsp, base_restart_strategy="first_choice")
    print(f"Random Restart (base: First Choice) Final tour cost: {candidate_restart_first_choice.value:.2f}")

    candidate_sa = local_search(TSPProblem(cities), strategy="simulated")
    print(f"Simulated Annealing Final tour cost: {candidate_sa.value:.2f}")
    print(f"(Best SA state found: {candidate_sa.state})")  # Show the state for SA

def test_eight_queens():
    print("\n----- Eight Queens Local Search Test -----")
    
    board_size = 8
    num_restarts_queens = iterations_queens = 50
    q_problem_base = EightQueensProblem(size=board_size)
    print(f"Sample Initial {board_size}-Queens state: {q_problem_base.initial_state}")
    print(f"Sample Initial conflicts: {q_problem_base.evaluate(q_problem_base.initial_state)}")

    # --- Test Hill Climbing Variants ---
    candidate_steepest = local_search(EightQueensProblem(size=board_size), strategy="steepest")
    print(f"Steepest Ascent Final conflicts: {candidate_steepest.value} (Goal reached? {candidate_steepest.value == 0})")

    candidate_stochastic = local_search(EightQueensProblem(size=board_size), strategy="stochastic")
    print(f"Stochastic HC Final conflicts: {candidate_stochastic.value} (Goal reached? {candidate_stochastic.value == 0})")

    candidate_first_choice = local_search(EightQueensProblem(size=board_size), strategy="first_choice")
    print(f"First Choice HC Final conflicts: {candidate_first_choice.value} (Goal reached? {candidate_first_choice.value == 0})")

    # --- Test Random Restart ---
    candidate_restart_steepest = local_search(EightQueensProblem(size=board_size), strategy="random_restart", num_restarts=num_restarts_queens, base_restart_strategy="steepest")
    print(f"Random Restart (base: Steepest) Final conflicts: {candidate_restart_steepest.value} (Goal reached? {candidate_restart_steepest.value == 0})")

    candidate_restart_stochastic = local_search(EightQueensProblem(size=board_size), strategy="random_restart", num_restarts=num_restarts_queens, base_restart_strategy="stochastic")
    print(f"Random Restart (base: Stochastic) Final conflicts: {candidate_restart_stochastic.value} (Goal reached? {candidate_restart_stochastic.value == 0})")

    candidate_restart_first_choice = local_search(EightQueensProblem(size=board_size), strategy="random_restart", num_restarts=num_restarts_queens, base_restart_strategy="first_choice")
    print(f"Random Restart (base: First Choice) Final conflicts: {candidate_restart_first_choice.value} (Goal reached? {candidate_restart_first_choice.value == 0})")

    candidate_sa = local_search(EightQueensProblem(size=board_size), strategy="simulated")
    print(f"Simulated Annealing Final conflicts: {candidate_sa.value} (Goal reached? {candidate_sa.value == 0})")
    if candidate_sa.value == 0:
         print(f"(SA Solution state: {candidate_sa.state})")

# ------------------------------------------------------------------------------
# Main Execution Block
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    # Optional: Seed random for reproducibility during testing

    test_tsp()
    test_eight_queens()



----- TSP Local Search Test -----
Sample Initial TSP tour: ['D', 'A', 'I', 'H', 'Q', 'C', 'L', 'B', 'K', 'R', 'M', 'G', 'S', 'P', 'N', 'J', 'E', 'O', 'F', 'T']
Sample Initial tour cost: 450.30
Steepest Ascent Final tour cost: 164.99
Stochastic HC Final tour cost: 503.91
First Choice HC Final tour cost: 181.26
Random Restart (base: Steepest) Final tour cost: 144.75
Random Restart (base: Stochastic) Final tour cost: 144.75
Random Restart (base: First Choice) Final tour cost: 144.75
Simulated Annealing Final tour cost: 164.12
(Best SA state found: ['L', 'J', 'H', 'I', 'F', 'G', 'E', 'C', 'A', 'B', 'D', 'K', 'M', 'R', 'T', 'S', 'Q', 'O', 'P', 'N'])

----- Eight Queens Local Search Test -----
Sample Initial 8-Queens state: [1, 0, 2, 7, 5, 6, 4, 3]
Sample Initial conflicts: 7
Steepest Ascent Final conflicts: 1 (Goal reached? False)
Stochastic HC Final conflicts: 4 (Goal reached? False)
First Choice HC Final conflicts: 0 (Goal reached? True)
Random Restart (base: Steepest) Final conflicts: 0